In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

In [2]:
df = pd.read_csv("../data/raw/resume_data.csv")

print(df.shape)

df.head()

(9544, 35)


,address,career_objective,skills,educational_institution_name,degree_names,passing_years,educational_results,result_types,major_field_of_studies,professional_company_names,...,online_links,issue_dates,expiry_dates,﻿job_position_name,educationaL_requirements,experiencere_requirement,age_requirement,responsibilities.1,skills_required,matched_score
0,NaN,Big data analytics working and database wareho...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...",['The Amity School of Engineering & Technology...,['B.Tech'],['2019'],['N/A'],[None],['Electronics'],['Coca-COla'],...,NaN,NaN,NaN,Senior Software Engineer,B.Sc in Computer Science & Engineering from a ...,At least 1 year,NaN,Technical Support\nTroubleshooting\nCollaborat...,NaN,0.850000
1,NaN,Fresher looking to join as a data analyst and ...,"['Data Analysis', 'Data Analytics', 'Business ...","['Delhi University - Hansraj College', 'Delhi ...","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']","['2015', '2018']","['N/A', 'N/A']","['N/A', 'N/A']","['Mathematics', 'Statistics']",['BIB Consultancy'],...,NaN,NaN,NaN,Machine Learning (ML) Engineer,M.Sc in Computer Science & Engineering or in a...,At least 5 year(s),NaN,Machine Learning Leadership\nCross-Functional ...,NaN,0.750000
2,NaN,NaN,"['Software Development', 'Machine Learning', '...","['Birla Institute of Technology (BIT), Ranchi']",['B.Tech'],['2018'],['N/A'],['N/A'],['Electronics/Telecommunication'],['Axis Bank Limited'],...,NaN,NaN,NaN,"Executive/ Senior Executive- Trade Marketing, ...",Master of Business Administration (MBA),At least 3 years,NaN,"Trade Marketing Executive\nBrand Visibility, S...",Brand Promotion\nCampaign Management\nField Su...,0.416667
3,NaN,To obtain a position in a fast-paced business ...,"['accounts payables', 'accounts receivables', ...","['Martinez Adult Education, Business Training ...",['Computer Applications Specialist Certificate...,['2008'],[None],[None],['Computer Applications'],"['Company Name ï¼ City , State', 'Company Name...",...,NaN,NaN,NaN,Business Development Executive,Bachelor/Honors,1 to 3 years,Age 22 to 30 years,Apparel Sourcing\nQuality Garment Sourcing\nRe...,Fast typing skill\nIELTSInternet browsing & on...,0.760000
4,NaN,Professional accountant with an outstanding wo...,"['Analytical reasoning', 'Compliance testing k...",['Kent State University'],['Bachelor of Business Administration'],[None],['3.84'],[None],['Accounting'],"['Company Name', 'Company Name', 'Company Name...",...,[None],[None],"['February 15, 2021']",Senior iOS Engineer,Bachelor of Science (BSc) in Computer Science,At least 4 years,NaN,iOS Lifecycle\nRequirement Analysis\nNative Fr...,iOS\niOS App Developer\niOS Application Develo...,0.650000


In [3]:
text_columns = [
    "career_objective",
    "skills",
    "major_field_of_studies",
    "related_skils_in_job",
    "positions",
    "responsibilities"
]

df["resume_text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

df["resume_text"].head()

0    Big data analytics working and database wareho...
1    Fresher looking to join as a data analyst and ...
2     ['Software Development', 'Machine Learning', ...
3    To obtain a position in a fast-paced business ...
4    Professional accountant with an outstanding wo...
Name: resume_text, dtype: str

In [4]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_resume"] = df["resume_text"].apply(clean_text)

df["clean_resume"].head()

0    big data analytics working and database wareho...
1    fresher looking to join as a data analyst and ...
2    software development machine learning deep lea...
3    to obtain a position in a fast paced business ...
4    professional accountant with an outstanding wo...
Name: clean_resume, dtype: str

In [5]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X = tfidf.fit_transform(df["clean_resume"])

print(X.shape)

(9544, 3578)


In [6]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=10,
    random_state=42,
    n_init=10
)

kmeans.fit(X)

print("✅ Clustering Completed")

✅ Clustering Completed


In [7]:
df["cluster"] = kmeans.labels_

df[["positions", "cluster"]].head()

,positions,cluster
0,['Big Data Analyst'],5
1,['Business Analyst'],2
2,['Software Developer (Machine Learning Enginee...,7
3,"['Accountant', 'Accounts Receivable Clerk', 'M...",8
4,"['Staff Accountant', 'Senior Accountant', 'Tax...",8


In [8]:
print(df["cluster"].value_counts())

cluster
1    2864
5    1846
0    1612
8    1150
4     599
3     400
6     286
2     280
9     274
7     233
Name: count, dtype: int64


In [9]:
for cluster in sorted(df["cluster"].unique()):
    print("=" * 50)
    print(f"Cluster {cluster}")

    print(
        df[df["cluster"] == cluster]["positions"]
        .value_counts()
        .head(10)
    )

Cluster 0
positions
['Intern']                              195
['Intern Trainee']                       67
['SDE']                                  46
['Junior Machine Learning Engineer']     45
['Analyst Intern']                       45
['Software Developer']                   44
['Machine Learning Engineer']            44
['Data Scientist']                       43
['SDE Intern']                           42
['Machine Learning Developer']           24
Name: count, dtype: int64
Cluster 1
positions
['Corporate Process/Systems Engineering Manager', 'SDI Site Operations Manager', 'Operations Group Manager']                                                                                                                                                            28
['QA Engineering Manager', 'Software Certification Analyst', 'Project Lead']                                                                                                                                                       